In [73]:
import numpy as np
import pandas as pd
import random
import time
from rapidfuzz import process, fuzz, distance
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import spoa 
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.cluster import HDBSCAN
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster, to_tree
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt

# --- 2. DATA GENERATION & UTILS ---
def mutate_sequence(seq, error_rate=0.10):
    if error_rate <= 0: return seq
    seq_list = list(seq)
    new_seq = []
    for base in seq_list:
        if random.random() < error_rate:
            r = random.random()
            if r < 0.5: new_seq.append(random.choice("ACGT")) 
            elif r < 0.75: 
                new_seq.append(base)
                new_seq.append(random.choice("ACGT"))
            else: pass 
        else:
            new_seq.append(base)
    return "".join(new_seq)

def gen_dna(k): return "".join(random.choices("ACGT", k=k))

# (Generating fresh data to ensure variables exist for the loop below)
n_items = 500
pool_bc = [gen_dna(12) for _ in range(n_items)]
pool_ins = [gen_dna(random.randint(300,400)) for _ in range(n_items)]
n_repeat_bc = 500
pool_bc += pool_bc[0:n_repeat_bc]
pool_ins += [gen_dna(random.randint(300,400)) for _ in range(n_repeat_bc)]
pool_bc += pool_bc[0:n_repeat_bc]
pool_ins += [gen_dna(random.randint(300,400)) for _ in range(n_repeat_bc)]

# Add very similar sequence that will defeat clustering!
pool_bc += [mutate_sequence(pool_bc[i], 0.03) for i in range(n_repeat_bc)]
pool_ins += [mutate_sequence(pool_ins[i], 0.03) for i in range(n_repeat_bc)]


data = []
for i in range(len(pool_bc)):
     n_reads = random.randint(8, 13) 
     for _ in range(n_reads):
         data.append({
             "ID": i,
             "Barcode": mutate_sequence(pool_bc[i], 0.05), 
             "Insert": mutate_sequence(pool_ins[i], 0.05)
         })

df = pd.DataFrame(data)

In [103]:
import time
import numpy as np
import pandas as pd
from scipy.stats import mode
from scipy.cluster.hierarchy import dendrogram
from sklearn.metrics import pairwise_distances
from sklearn.cluster import AgglomerativeClustering
import spoa
from numba import njit, prange
# Assuming rapidfuzz or fuzzywuzzy is available based on context
try:
    from rapidfuzz import process, fuzz
except ImportError:
    from fuzzywuzzy import process, fuzz

# ==========================================
# TIMING UTILITY
# ==========================================
class GlobalTimer:
    def __init__(self):
        self.timings = {}
        self.starts = {}

    def start(self, name):
        if name not in self.starts: # Don't overwrite if nested recursion causes restart
            self.starts[name] = time.perf_counter()

    def stop(self, name):
        if name in self.starts:
            elapsed = time.perf_counter() - self.starts[name]
            self.timings[name] = self.timings.get(name, 0) + elapsed
            del self.starts[name]

    def report(self):
        print("\n" + "="*40)
        print(f"{'OPERATION':<30} | {'TIME (s)':<10}")
        print("-" * 43)
        for name, val in sorted(self.timings.items(), key=lambda x: x[1], reverse=True):
            print(f"{name:<30} | {val:.4f}")
        print("="*40 + "\n")

timer = GlobalTimer()

# ==========================================
# CORE FUNCTIONS
# ==========================================

def mutate_integers_fast(seed_int_arr, error_rate, n_copies, mutation_choices):
    """
    Mutates an integer array directly using NumPy. 
    Orders of magnitude faster than string manipulation.
    """
    L = len(seed_int_arr)
    
    # 1. Tile the seed (Create N copies)
    copies = np.tile(seed_int_arr, (n_copies, 1))
    
    # 2. Generate Error Mask
    mask = np.random.random((n_copies, L)) < error_rate
    
    # 3. Apply Mutations
    n_mutations = np.sum(mask)
    if n_mutations > 0:
        replacements = np.random.choice(mutation_choices, size=n_mutations)
        copies[mask] = replacements
        
    return copies


def encode_msa(msa_strings, alphabet="ACGTN-"):
    timer.start("encode_msa")
    """
    Vectorized encoding using ASCII byte views. 
    Assumes inputs are ASCII (standard for DNA).
    """
    N = len(msa_strings)
    if N == 0: 
        timer.stop("encode_msa")
        return np.array([]), len(alphabet)
    
    max_len = max(len(s) for s in msa_strings)
    
    # 1. Pad and convert to single byte-string buffer
    padded_block = "".join([s.ljust(max_len, '-') for s in msa_strings]).encode('ascii')
    
    # 2. View as NumPy int8 array directly
    arr_view = np.frombuffer(padded_block, dtype=np.int8).copy()
    arr_view = arr_view.reshape(N, max_len)
    
    # 3. Fast Translation Table
    lookup = np.zeros(256, dtype=np.int8) + (len(alphabet) - 1) 
    for idx, char in enumerate(alphabet):
        lookup[ord(char)] = idx
        
    # 4. Apply translation
    msa_int = lookup[arr_view]
    
    timer.stop("encode_msa")
    return msa_int, len(alphabet)


def get_marginals(msa_int, vocab_size):
    # Create on hot matrix
    one_hot = np.eye(vocab_size)[msa_int]
    # Calculate frequencies of each base/N/- at each position in MSA
    P_i = one_hot.mean(axis=0)
    return one_hot, P_i

@njit(parallel=True, fastmath=True)
def compute_mi_numba(msa_int, vocab_size):
    N, L = msa_int.shape
    A = vocab_size
    mi = np.zeros((L, L), dtype=np.float32)

    # per-column marginals
    P_i = np.zeros((L, A), dtype=np.float32)
    for i in range(L):
        counts = np.zeros(A, dtype=np.int32)
        for r in range(N):
            counts[msa_int[r, i]] += 1
        for a in range(A):
            P_i[i, a] = counts[a] / N

    # joint and MI
    for i in prange(L):
        for j in range(i+1, L):
            counts = np.zeros((A, A), dtype=np.int32)
            for r in range(N):
                counts[msa_int[r, i], msa_int[r, j]] += 1
            mi_val = 0.0
            for a in range(A):
                for b in range(A):
                    nij = counts[a, b]
                    if nij == 0:
                        continue
                    pij = nij / N
                    pprod = P_i[i, a] * P_i[j, b]
                    if pprod > 0:
                        mi_val += pij * np.log(pij / pprod)
            mi[i, j] = mi_val
            mi[j, i] = mi_val
    return mi


def compute_mi_scores_optimized(one_hot, P_i):
    # NOTE: The one_hot logic here is actually bypassed if we use the msa_int directly 
    # in the numba function above. 
    # However, to keep your structure valid, we assume msa_int is passed or 
    # we reconstruct it. 
    # Actually, your 'compute_mi_numba' takes 'msa_int', but this wrapper takes 'one_hot'.
    # This wrapper seems disconnected from the Numba function signature you provided.
    # I will adapt this to work with the Numba function by converting one_hot back to int
    # or assuming the caller fixes it. 
    
    # For timing purposes, I will just time the Numba execution.
    timer.start("compute_mi_numba_execution")
    
    # Hack to get int array back from one hot for the Numba function
    # (In a real optimization, pass msa_int directly to this function)
    msa_int = np.argmax(one_hot, axis=2) 
    vocab_size = one_hot.shape[2]
    
    result = compute_mi_numba(msa_int, vocab_size)
    timer.stop("compute_mi_numba_execution")
    return result


def get_elbow_columns(mi_matrix, exclusion_distance=5, verbose=0):
    timer.start("get_elbow_columns")
    L = mi_matrix.shape[0]
    mask = np.triu(np.ones((L, L), dtype=bool), k=exclusion_distance + 1)
    rows, cols = np.where(mask)
    scores = mi_matrix[rows, cols]
    
    if len(scores) == 0: 
        timer.stop("get_elbow_columns")
        return np.array([])

    # 1. Sort Descending
    sorted_indices = np.argsort(scores)[::-1]
    sorted_scores = scores[sorted_indices]
    
    # 2. Determine "Signal End" (Noise Floor Truncation)
    noise_floor = np.percentile(scores, 25)
    valid_mask = sorted_scores > noise_floor
    n_signal_points = np.sum(valid_mask)
    min_points = min(len(sorted_scores), 5)
    cutoff_idx = max(n_signal_points, min_points)
    
    # 3. Truncate for Geometry Calculation
    curve_y = sorted_scores[:cutoff_idx]
    n_points = len(curve_y)
    
    if n_points < 3:
        timer.stop("get_elbow_columns")
        return np.unique(np.concatenate([rows[sorted_indices[:n_points]], cols[sorted_indices[:n_points]]]))

    # 4. Standard Kneedle on Truncated Curve
    x_norm = np.linspace(0, 1, n_points)
    y_norm = (curve_y - curve_y.min()) / (curve_y.max() - curve_y.min() + 1e-9)
    
    line_vec = np.array([1.0, -1.0]) 
    line_vec = line_vec / np.linalg.norm(line_vec)
    vec_from_start = np.stack([x_norm, y_norm - 1.0], axis=1)
    distances = np.abs(vec_from_start[:, 0] * line_vec[1] - vec_from_start[:, 1] * line_vec[0])
    
    # 5. Select
    elbow_idx = np.argmax(distances)
    if elbow_idx == 0:
        for i in range(1, n_points - 1):
            if curve_y[i] >= (curve_y[0] * 0.95): 
                elbow_idx = i
            else:
                break

    n_selected = elbow_idx + 1 
    selected_indices = sorted_indices[:n_selected]
    selected_columns = np.unique(np.concatenate([rows[selected_indices], cols[selected_indices]]))
    
    timer.stop("get_elbow_columns")
    return selected_columns


def calculate_msa_mi_and_top_cols(barcodes, inserts, verbose=0, mi_exclusion_distance=3):
    
    timer.start("SPOA_alignment_initial")
    _, msa_barcodes = spoa.poa(barcodes, algorithm=2) 
    _, msa_inserts = spoa.poa(inserts, algorithm=2)
    timer.stop("SPOA_alignment_initial")

    msa_strings = [b + "-"*1 + i for b, i in zip(msa_barcodes, msa_inserts)]
    len_msa_inserts = len(msa_inserts[0]) 

    msa_int, vocab_size = encode_msa(msa_strings)
    one_hot, P_i = get_marginals(msa_int, vocab_size)
    
    mi_matrix = compute_mi_scores_optimized(one_hot, P_i)

    top_cols = get_elbow_columns(mi_matrix, mi_exclusion_distance, verbose=verbose)   

    return msa_int, mi_matrix, top_cols, len_msa_inserts


def plot_dendrogram(model, **kwargs):
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1 
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack([model.children_, model.distances_, counts]).astype(float)
    dendrogram(linkage_matrix, **kwargs)


def cluster_msa_subset(msa_subset, expected_error_rate, error_rate_multiplier=None, jump_thresh=None):
    assert (error_rate_multiplier is None) + (jump_thresh is None) == 1
    
    timer.start("hamming_distance_matrix")
    dist_matrix_subset = pairwise_distances(msa_subset, metric='hamming')
    timer.stop("hamming_distance_matrix")

    if dist_matrix_subset.shape[0] == 1:
        return dist_matrix_subset, np.array([0])
    
    if error_rate_multiplier is not None: 
        d_thresh = expected_error_rate * error_rate_multiplier
    else:
        temp_matrix = dist_matrix_subset.copy() 
        np.fill_diagonal(temp_matrix, np.inf) 
        min_dist = np.min(temp_matrix)
        d_thresh = min_dist * jump_thresh + expected_error_rate

    timer.start("agglomerative_clustering")
    agg = AgglomerativeClustering(
        metric="precomputed",
        linkage="single",
        distance_threshold = d_thresh,
        n_clusters=None)
    
    labels = agg.fit_predict(dist_matrix_subset) 
    timer.stop("agglomerative_clustering")

    return dist_matrix_subset, labels


def cluster_barcode_insert_pairs(barcodes, inserts, mi_exclusion_distance, expected_error_rate, 
                                 error_rate_multiplier=None, jump_thresh=None, verbose=0, subset_msa=True):
    
    msa_int, mi_matrix, top_cols, len_msa_inserts = calculate_msa_mi_and_top_cols(barcodes, inserts, verbose, mi_exclusion_distance)

    if subset_msa:
        msa_subset = msa_int[:, top_cols]
    else:
        msa_subset = msa_int 

    dist_matrix_subset, labels = cluster_msa_subset(msa_subset, expected_error_rate, error_rate_multiplier, jump_thresh)

    return msa_int, mi_matrix, msa_subset, dist_matrix_subset, labels, len_msa_inserts


def fast_consensus(msa_list, threshold=0.5):
    if not msa_list: return ""
    arr = np.array([list(s) for s in msa_list], dtype='S1')
    arr_int = arr.view(np.int8)
    n_seqs, length = arr_int.shape
    consensus_int = []
    GAP_ASCII = 45 
    TO_LOWER_OFFSET = 32 
    
    for i in range(length):
        col = arr_int[:, i]
        vals, counts = np.unique(col, return_counts=True)
        max_count = counts.max()
        winners = vals[counts == max_count]
        best_char_code = winners[0] 
        is_ambiguous = len(winners) > 1
        
        if best_char_code != GAP_ASCII:
            if (max_count / n_seqs) >= threshold:
                if is_ambiguous:
                    if 65 <= best_char_code <= 90: 
                        best_char_code += TO_LOWER_OFFSET
                consensus_int.append(best_char_code)

    return np.array(consensus_int, dtype=np.uint8).view('S1').tobytes().decode('utf-8')


def get_poa_consensus(barcode_series, bias_towards=None):
    timer.start("SPOA_Consensus_Total")
    barcode_list = barcode_series.tolist()
    if bias_towards is not None:
        barcode_list.append(bias_towards)
    
    _, msa = spoa.poa(barcode_list, algorithm=1)
    consensus = fast_consensus(msa) 
    timer.stop("SPOA_Consensus_Total")
    return consensus 

    
def recursive_outlier_removal(all_df, cluster_df, filtered_dist_matrix, expected_error_rate, percentile_th, 
                              verbose, error_rate_multiplier=1.2, n_decoys=50):
    
    if len(cluster_df) < 2:
        return cluster_df

    m = filtered_dist_matrix.astype(float)
    m[np.triu_indices_from(m)] = np.nan
    max_val = np.nanmax(m)

    if max_val <= expected_error_rate*error_rate_multiplier: 
        return cluster_df

    m = filtered_dist_matrix.astype(float)
    m[np.triu_indices_from(m)] = np.nan
    median_dist = np.nanmedian(m)

    agg = AgglomerativeClustering(
        metric="precomputed",
        linkage="single",
        distance_threshold=median_dist,
        n_clusters=None 
    )

    cluster_labels = agg.fit_predict(filtered_dist_matrix)
    unique, counts = np.unique(cluster_labels, return_counts=True)
    largest_cluster_label = unique[np.argmax(counts)]
    
    largest_cluster_indices = np.where(cluster_labels == largest_cluster_label)[0]
    largest_cluster_df = cluster_df.iloc[largest_cluster_indices] 
    largest_subclust_dist_matrix = filtered_dist_matrix[np.ix_(largest_cluster_indices, largest_cluster_indices)]

    mean_distances = np.mean(largest_subclust_dist_matrix, axis=1)
    core_member_index = np.argmin(mean_distances)
    best_core_insert = largest_cluster_df['Insert'].iloc[core_member_index]

    # --- TIMING CRITICAL SECTION ---
    timer.start("Fuzzy_CDIST_Calculations")
    all_ratios = process.cdist([best_core_insert], list(cluster_df['Insert']), scorer=fuzz.ratio, dtype=np.float32)[0]
    timer.stop("Fuzzy_CDIST_Calculations")

    outlier_position = np.argmin(all_ratios) 
    outlier_ratio = all_ratios[outlier_position]
    outlier_insert = cluster_df['Insert'].iloc[outlier_position]

    random_inserts = all_df['Insert'].sample(n=n_decoys)

    # --- TIMING CRITICAL SECTION ---
    timer.start("Fuzzy_CDIST_Calculations")
    decoy_ratios = process.cdist([outlier_insert], random_inserts, scorer=fuzz.ratio, dtype=np.float32)[0]
    timer.stop("Fuzzy_CDIST_Calculations")

    threshold_ratio = np.percentile(decoy_ratios, percentile_th)

    if outlier_ratio >= threshold_ratio:
        return cluster_df 
    else:
        keep_mask = np.arange(len(filtered_dist_matrix)) != outlier_position
        new_dist_matrix = filtered_dist_matrix[np.ix_(keep_mask, keep_mask)]
        new_cluster_df = cluster_df.iloc[keep_mask]
        
        return recursive_outlier_removal(all_df, new_cluster_df, 
                                         new_dist_matrix, 
                                         expected_error_rate, percentile_th, verbose, error_rate_multiplier, n_decoys)

def calc_max_row_ratio(mi_matrix, top_pct=0.05, bottom_pct=0.50):
    n_sites = mi_matrix.shape[1]
    if n_sites < 5: return 0.0 
    
    mi_matrix = np.maximum(mi_matrix, 0)
    sorted_matrix = np.sort(mi_matrix, axis=1)
    k_top = max(1, int(n_sites * top_pct))      
    k_bottom = max(1, int(n_sites * bottom_pct))
    
    top_mean = np.mean(sorted_matrix[:, -k_top:], axis=1)
    bottom_mean = np.mean(sorted_matrix[:, :k_bottom], axis=1)
    
    damping_factor = 0.01
    ratios = (top_mean + damping_factor) / (bottom_mean + damping_factor)
    
    return np.max(ratios)


def check_cluster_integrity_mi(cluster_df, expected_error_rate, verbose=0):
    timer.start("MI_Simulations")
    barcodes = cluster_df['Barcode'].tolist()
    inserts = cluster_df['Insert'].tolist()
    n_members = len(barcodes)
    
    if n_members < 10: 
        timer.stop("MI_Simulations")
        return False
    
    # --- A. Alignment & Initial Encoding ---
    timer.start("SPOA_alignment_integrity_check")
    _, msa_barcodes = spoa.poa(barcodes, algorithm=1) 
    _, msa_inserts = spoa.poa(inserts, algorithm=1)
    timer.stop("SPOA_alignment_integrity_check")

    msa_strings = [b + "-"*1 + i for b, i in zip(msa_barcodes, msa_inserts)]
    
    msa_int, vocab_size = encode_msa(msa_strings)
    
    alphabet="ACGTN-"
    char_to_int = {c: i for i, c in enumerate(alphabet)}
    valid_mutation_indices = np.array([char_to_int[c] for c in "ACGT"])
    
    # --- B. Calculate Real Score ---
    one_hot, P_i = get_marginals(msa_int, vocab_size)
    mi_matrix_real = compute_mi_scores_optimized(one_hot, P_i)
    real_score = calc_max_row_ratio(mi_matrix_real)
    
    # --- C. Simulation Setup ---
    seed_idx = np.random.randint(n_members)
    seed_int = msa_int[seed_idx] 
    
    stages = [5, 15, 80]
    current_sim_scores = []
    total_sims_run = 0
    max_sim_score = 0.0 
    
    # --- D. The Fast Loop ---
    for step_count in stages:
        
        for _ in range(step_count):
            synthetic_ints = mutate_integers_fast(seed_int, expected_error_rate, n_members, valid_mutation_indices)
            s_one_hot = np.eye(vocab_size)[synthetic_ints]
            s_P_i = s_one_hot.mean(axis=0)
            
            # NOTE: calling the optimized version
            s_mi = compute_mi_scores_optimized(s_one_hot, s_P_i)
            score = calc_max_row_ratio(s_mi)
            
            current_sim_scores.append(score)
            if score > max_sim_score: max_sim_score = score
            
        total_sims_run += step_count
        
        if real_score <= max_sim_score:
            timer.stop("MI_Simulations")
            return False
            
    # --- E. Final Statistical Check ---
    threshold_99 = np.percentile(current_sim_scores, 99)
    
    if real_score > threshold_99:
        if verbose >= 1: 
            print(f"!!! WARNING: Cluster MI Outlier Detected!")
        timer.stop("MI_Simulations")
        return True
        
    timer.stop("MI_Simulations")
    return False
    

def full_analysis(sub_df, all_df, barcode_target, percentile_th, expected_error_rate, 
                  verbose, mi_exclusion_distance, cluster_again_total_mean_thresh=2,
                  cluster_again_single_mean_thresh=3):

    if len(sub_df) < 2:
        if 'cluster' not in sub_df.columns: sub_df = sub_df.copy(); sub_df['cluster'] = -1
        return sub_df

    barcodes = sub_df['Barcode'].tolist()
    inserts = sub_df['Insert'].tolist()
    
    msa_int, mi_matrix, msa_subset, dist_matrix_subset, labels, len_msa_inserts = cluster_barcode_insert_pairs(
                                                                            barcodes, 
                                                                            inserts, 
                                                                            mi_exclusion_distance,
                                                                            expected_error_rate,
                                                                            error_rate_multiplier=2,
                                                                            verbose=0,
                                                                            subset_msa=False)

    labels = labels + hash(frozenset(sub_df.index)) % 10_000_000
    sub_df['cluster'] = labels

    cluster_results = []

    for label in list(set(labels)):
        this_cluster = sub_df[sub_df['cluster'] == label]
        indices = np.where(labels == label)[0]
        filtered_dist_matrix = dist_matrix_subset[np.ix_(indices, indices)]

        final_processed_cluster = recursive_outlier_removal(all_df, this_cluster, filtered_dist_matrix, 
                                                            expected_error_rate, percentile_th, verbose)

        print("Checking MI")
        if len(final_processed_cluster) >= 8 and label != -1:
            is_suspicious = check_cluster_integrity_mi(final_processed_cluster, expected_error_rate, verbose=verbose)
            
            final_processed_cluster = final_processed_cluster.copy()
            final_processed_cluster['mi_warning'] = is_suspicious
            
            if is_suspicious:
                print(f"Cluster {label} may be formed of two highly similar sequences")
        else:
            final_processed_cluster = final_processed_cluster.copy()
            final_processed_cluster['mi_warning'] = False
                
        cluster_results.append(final_processed_cluster)

    result = pd.concat(cluster_results)
    
    # === PRINT TIMING REPORT ===
    timer.report()
    
    return result

def filter_result_df(result, barcode_target, verbose):
    result = result.copy()    

    # Optimized consensus calculation to avoid re-calculating same cluster multiple times
    unique_clusters = result['cluster'].unique()
    consensus_map = {}
    
    timer.start("filter_consensus_generation")
    for cl in unique_clusters:
        subset = result.loc[result['cluster'] == cl, 'Barcode']
        consensus_map[cl] = get_poa_consensus(subset)
    timer.stop("filter_consensus_generation")

    result['cluster_consensus_bc'] = result['cluster'].map(consensus_map)

    result = result[
        (result['cluster_consensus_bc'] == barcode_target) | 
        ( (result.groupby('cluster')['Barcode'].transform('size') == 2) & 
          (result.groupby('cluster')['Barcode'].transform(lambda x: (x == barcode_target).any())) 
        )
    ]

    result['cluster_consensus_insert'] = result.groupby('cluster')['Insert'].transform(get_poa_consensus)

    unique = [u for u in sorted(result['cluster'].unique()) if u != -1]
    mapping = {old: new for new, old in enumerate(unique)}
    result['cluster'] = result['cluster'].map(lambda x: mapping.get(x, -1))

    return result

In [ ]:
# ==== PARAMS =====
from sklearn.manifold import MDS
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import MDS
import numpy as np
from sklearn.cluster import OPTICS
from sklearn.cluster import SpectralClustering
from scipy.linalg import eigh
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.cluster import AffinityPropagation
import numpy as np
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram
from scipy.optimize import curve_fit

verbose = 1
percentile_th = 95
expected_error_rate = 0.05
PLOTS = False
mi_exclusion_distance=3
n_sims = 150
sim_percentile = 0.1

# ==== RUN =====

barcode_counts = df['Barcode'].value_counts()
all_barcodes = np.array(barcode_counts.index.tolist())     
    

for i, (top_bc, count) in enumerate(barcode_counts.head(10).items()):

    if verbose >= 1:
        print(f"\n\n=============\n{top_bc}")
        print(f"Processing Top Barcode #{i+1}: {top_bc} (Count: {count})")

    # top_bc = "GCAATGCTTGTC" # Force specific barcode if needed for debug
    
    # RapidFuzz
    scores = process.cdist([top_bc], all_barcodes, scorer=fuzz.ratio, dtype=np.float32)[0]
    candidate_bcs = all_barcodes[np.where(scores > 85)[0]]
    filtered_df = df[df['Barcode'].isin(candidate_bcs)].copy()

    # filtered_df = pd.read_csv("~/Downloads/fml.csv")

    if filtered_df.empty: continue

    if verbose >= 2:
        print(f"  > RapidFuzz gathered {len(filtered_df)} reads")

    result_df = full_analysis(filtered_df, df, top_bc, 
                                          percentile_th=percentile_th, verbose=verbose, expected_error_rate=expected_error_rate,
                             mi_exclusion_distance=mi_exclusion_distance)

    result_df2 = filter_result_df(result_df, top_bc, verbose)

    if verbose >= 1:
        print(result_df2[['ID', 'cluster', 'mi_warning']])



CCACCCAGGAAG
Processing Top Barcode #1: CCACCCAGGAAG (Count: 35)
Checking MI
Checking MI
Checking MI
!!! WARNING: Cluster MI Outlier Detected!
Cluster 7351662 may be formed of two highly similar sequences
Checking MI

OPERATION                      | TIME (s)  
-------------------------------------------
compute_mi_numba_execution     | 5.7680
MI_Simulations                 | 4.9208
SPOA_alignment_initial         | 0.0740
SPOA_alignment_integrity_check | 0.0299
get_elbow_columns              | 0.0184
hamming_distance_matrix        | 0.0018
encode_msa                     | 0.0016
Fuzzy_CDIST_Calculations       | 0.0007
agglomerative_clustering       | 0.0007

         ID  cluster  mi_warning
9213    878        2       False
9214    878        2       False
9215    878        2       False
9216    878        2       False
9218    878        2       False
9219    878        2       False
9220    878        2       False
9221    878        2       False
14500  1378        0       False
1

In [5]:
result_df2

,ID,Barcode,Insert,cluster,cluster_consensus_bc,cluster_consensus_insert
125060,20793,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCAGCAGAATGGAC...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125061,20793,GGTATACATAGT,ATCGGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACTC...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125062,20793,GGTATTACATAGT,AACGCGACGTACCAAACATACCAACGGGTCTATCACAGAATGGACT...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125063,20793,GGTATACATAGT,AACGCGACGTGCCAAAACTACCAACGGGTCTATCATCAGCATGGAC...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125064,20793,GGGTATTACATAGT,GACGAGACGTACCAAAACTACCAACGGGTCTATTACAGAATGGACT...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125065,20793,GGTATACATAGT,AACGCGACGTCACCAAATACTCCAACGGGTCTATCACAGAATGGCT...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125066,20793,GGTATACATTAGT,AACGCGACGTACCAAAACTTACCAACGGGTCTATCACAGAATGGAC...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125067,20793,GGTATACATAGT,AACGCCGACGAACCAAAACTACCAACGGGTCTATCACAGAATGGAC...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125068,20793,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCAAAGAATGGACT...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
121972,20293,GGTATACATAGT,CTTGCTCTGACAACAGACATAAACGGAGTAATCTTAGTGCTGTCTG...,1,GGTATACATAGT,CTTGCTCTGACAACAGACATAAACGGAGTAATCTTAGTGCTGTTGG...


In [109]:
result_df2

,ID,Barcode,Insert,cluster,cluster_consensus_bc,cluster_consensus_insert
347,56,TCGCGCACAGGA,TTGGTAACGCTCTGCGACCAATAACGCGGTGCACCAGCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
348,56,TCGCGACAGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCA,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
349,56,CCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
350,56,TCGCGAACAGGA,TTGGTACGCACTGCGAAATAACGCGGTGTCGTCAGCGTCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
351,56,TCGCGAACAGGA,TTGGTACGCTCTGTGACCAATAACGCGGTGCATCACGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
352,56,TCGCTAAACAGGA,TTGGTACGCTTGCGAGCCAATAACGCGATGCATCAGCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
353,56,TCGCGAACAGGA,TTGGTACGCTGTGCGACAATAACGCGGTGATCAGCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
354,56,TCGCGAACAGGA,TTGGTACGCTCTCGCGCCAATAACGCGGTGCATCAGCTGCAT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
355,56,TCGCGAACAGGA,TTGGTCCGCTCTGCGACCAATAACGCAGTGCATCAGCCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
356,56,TCGCGAACAGGA,TTGGTACTCTCTACGACCAATAACGCGGTGAATCTGTGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
